In [ ]:
#해당 파일에서는 데이터 품질/구조 검증을 진행

In [1]:
!pip install duckdb

   ---------------------------------------- 0.0/13.2 MB ? eta -:--:--
   ------- -------------------------------- 2.6/13.2 MB 12.5 MB/s eta 0:00:01
   ----------- ---------------------------- 3.9/13.2 MB 9.0 MB/s eta 0:00:02
   --------------------- ------------------ 7.1/13.2 MB 11.2 MB/s eta 0:00:01
   ------------------------------- -------- 10.5/13.2 MB 12.3 MB/s eta 0:00:01
   ---------------------------------------- 13.2/13.2 MB 12.5 MB/s eta 0:00:00


In [3]:
import duckdb

print(duckdb.__version__)

1.5.5


In [ ]:
'''
event_time	    사용자의 행동이 발생한 시간
event_type	    행동 종류 (view, cart, remove_from_cart, purchase)
product_id	    행동 대상이 된 상품의 고유 ID
category_id	    상품이 속한 카테고리의 고유 ID
category_code	사람이 이해하기 쉬운 상품 카테고리명/분류 (electronics.smartphone 등)
brand	        상품 브랜드명
price	        해당 이벤트 시점의 상품 가격
user_id	        사용자를 구분하는 고유 ID
user_session	한 번의 방문/활동 세션을 구분하는 ID
'''

In [9]:
csv_path = r"..\data\raw\2019-Oct.csv"

duckdb.sql(f"""
    SELECT *
    FROM read_csv_auto('{csv_path}')
    LIMIT 5
""").show()

┌─────────────────────┬────────────┬────────────┬─────────────────────┬─────────────────────────────────────┬──────────┬─────────┬───────────┬──────────────────────────────────────┐
│     event_time      │ event_type │ product_id │     category_id     │            category_code            │  brand   │  price  │  user_id  │             user_session             │
│      timestamp      │  varchar   │   int64    │        int64        │               varchar               │ varchar  │ double  │   int64   │               varchar                │
├─────────────────────┼────────────┼────────────┼─────────────────────┼─────────────────────────────────────┼──────────┼─────────┼───────────┼──────────────────────────────────────┤
│ 2019-10-01 00:00:00 │ view       │   44600062 │ 2103807459595387724 │ NULL                                │ shiseido │   35.79 │ 541312140 │ 72d76fde-8bb3-4e00-8c23-a032dfed738c │
│ 2019-10-01 00:00:00 │ view       │    3900821 │ 2053013552326770905 │ appliances.environ

In [11]:
#10월 데이터 전체 행 수
duckdb.sql(f"""
    SELECT COUNT(*) AS row_count
    FROM read_csv_auto('{csv_path}')
""").show()

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

┌───────────┐
│ row_count │
│   int64   │
├───────────┤
│  42448764 │
└───────────┘



In [13]:
#실제로 10월 1일~31일 데이터가 맞는지 확인
duckdb.sql(f"""
    SELECT 
        MIN(event_time) AS min_time,
        MAX(event_time) AS max_time
    FROM read_csv_auto('{csv_path}')
""").show()

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

┌─────────────────────┬─────────────────────┐
│      min_time       │      max_time       │
│      timestamp      │      timestamp      │
├─────────────────────┼─────────────────────┤
│ 2019-10-01 00:00:00 │ 2019-10-31 23:59:59 │
└─────────────────────┴─────────────────────┘



In [15]:
#view, cart, purchase 등 행동 유형별 이벤트 개수 확인
duckdb.sql(f"""
    SELECT 
        event_type,
        COUNT(*) AS event_count
    FROM read_csv_auto('{csv_path}')
    GROUP BY event_type
    ORDER BY event_count DESC
""").show()

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

┌────────────┬─────────────┐
│ event_type │ event_count │
│  varchar   │    int64    │
├────────────┼─────────────┤
│ view       │    40779399 │
│ cart       │      926516 │
│ purchase   │      742849 │
└────────────┴─────────────┘



In [17]:
#고유 사용자 수와 고유 세션 수 확인
duckdb.sql(f"""
    SELECT
        COUNT(DISTINCT user_id) AS unique_users,
        COUNT(DISTINCT user_session) AS unique_sessions
    FROM read_csv_auto('{csv_path}')
""").show()

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

┌──────────────┬─────────────────┐
│ unique_users │ unique_sessions │
│    int64     │      int64      │
├──────────────┼─────────────────┤
│      3022290 │         9244421 │
└──────────────┴─────────────────┘



In [19]:
#category_code, brand, user_id, user_session 등 주요 컬럼의 결측치 규모 확인
duckdb.sql(f"""
    SELECT
        SUM(CASE WHEN category_code IS NULL THEN 1 ELSE 0 END) AS category_code_null,
        SUM(CASE WHEN brand IS NULL THEN 1 ELSE 0 END) AS brand_null,
        SUM(CASE WHEN user_id IS NULL THEN 1 ELSE 0 END) AS user_id_null,
        SUM(CASE WHEN user_session IS NULL THEN 1 ELSE 0 END) AS user_session_null
    FROM read_csv_auto('{csv_path}')
""").show()

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

┌────────────────────┬────────────┬──────────────┬───────────────────┐
│ category_code_null │ brand_null │ user_id_null │ user_session_null │
│       int128       │   int128   │    int128    │      int128       │
├────────────────────┼────────────┼──────────────┼───────────────────┤
│           13515609 │    6113008 │            0 │                 2 │
└────────────────────┴────────────┴──────────────┴───────────────────┘



In [21]:
# 확인 결과
# - 이벤트 유형: view, cart, purchase 3종
# - 고유 사용자 약 302만 명, 고유 세션 약 924만 개
# - user_id 결측 없음, user_session 결측 2건 → 사용자/세션 분석에 큰 문제 없음
# - category_code 결측 약 31.8%, brand 결측 약 14.4% → 상품 속성 분석 시 주의 필요

In [23]:
# product_id, category_id, price 핵심 상품 컬럼의 결측치와 가격 범위 확인
duckdb.sql(f"""
    SELECT
        SUM(CASE WHEN product_id IS NULL THEN 1 ELSE 0 END) AS product_id_null,
        SUM(CASE WHEN category_id IS NULL THEN 1 ELSE 0 END) AS category_id_null,
        SUM(CASE WHEN price IS NULL THEN 1 ELSE 0 END) AS price_null,
        MIN(price) AS min_price,
        MAX(price) AS max_price
    FROM read_csv_auto('{csv_path}')
""").show()

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

┌─────────────────┬──────────────────┬────────────┬───────────┬───────────┐
│ product_id_null │ category_id_null │ price_null │ min_price │ max_price │
│     int128      │      int128      │   int128   │  double   │  double   │
├─────────────────┼──────────────────┼────────────┼───────────┼───────────┤
│               0 │                0 │          0 │       0.0 │   2574.07 │
└─────────────────┴──────────────────┴────────────┴───────────┴───────────┘



In [25]:
# 확인 결과
# - product_id, category_id, price 결측치 없음
# - 상품/카테고리/가격 기반 분석 가능
# - 최소 가격이 0이므로 0원 상품 데이터는 추가 확인 필요

In [28]:
# 11월 데이터 경로 설정
nov_path = r"..\data\raw\2019-Nov.csv"

# 11월 데이터의 컬럼 구조와 샘플 5행 확인
duckdb.sql(f"""
    SELECT *
    FROM read_csv_auto('{nov_path}')
    LIMIT 5
""").show()

┌─────────────────────┬────────────┬────────────┬─────────────────────┬───────────────────────────┬─────────┬────────┬───────────┬──────────────────────────────────────┐
│     event_time      │ event_type │ product_id │     category_id     │       category_code       │  brand  │ price  │  user_id  │             user_session             │
│      timestamp      │  varchar   │   int64    │        int64        │          varchar          │ varchar │ double │   int64   │               varchar                │
├─────────────────────┼────────────┼────────────┼─────────────────────┼───────────────────────────┼─────────┼────────┼───────────┼──────────────────────────────────────┤
│ 2019-11-01 00:00:00 │ view       │    1003461 │ 2053013555631882655 │ electronics.smartphone    │ xiaomi  │ 489.07 │ 520088904 │ 4d3b30da-a5e4-49df-b1a8-ba5943f1dd33 │
│ 2019-11-01 00:00:00 │ view       │    5000088 │ 2053013566100866035 │ appliances.sewing_machine │ janome  │ 293.65 │ 530496790 │ 8e5f4f83-366c-4f70-

In [30]:
# 11월 전체 이벤트 수와 실제 데이터 기간 확인
duckdb.sql(f"""
    SELECT
        COUNT(*) AS row_count,
        MIN(event_time) AS min_time,
        MAX(event_time) AS max_time
    FROM read_csv_auto('{nov_path}')
""").show()

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

┌───────────┬─────────────────────┬─────────────────────┐
│ row_count │      min_time       │      max_time       │
│   int64   │      timestamp      │      timestamp      │
├───────────┼─────────────────────┼─────────────────────┤
│  67501979 │ 2019-11-01 00:00:00 │ 2019-11-30 23:59:59 │
└───────────┴─────────────────────┴─────────────────────┘



In [32]:
# 11월에도 view, cart, purchase의 동일한 이벤트 구조인지 확인
duckdb.sql(f"""
    SELECT
        event_type,
        COUNT(*) AS event_count
    FROM read_csv_auto('{nov_path}')
    GROUP BY event_type
    ORDER BY event_count DESC
""").show()

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

┌────────────┬─────────────┐
│ event_type │ event_count │
│  varchar   │    int64    │
├────────────┼─────────────┤
│ view       │    63556110 │
│ cart       │     3028930 │
│ purchase   │      916939 │
└────────────┴─────────────┘



In [34]:
# 확인 결과
# - 11월도 view, cart, purchase 3종 이벤트 구조
# - 10월과 동일한 퍼널 구조로 분석 가능

In [36]:
# 11월 전체 행 수와 실제 데이터 기간 확인
duckdb.sql(f"""
    SELECT
        COUNT(*) AS row_count,
        MIN(event_time) AS min_time,
        MAX(event_time) AS max_time
    FROM read_csv_auto('{nov_path}')
""").show()

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

┌───────────┬─────────────────────┬─────────────────────┐
│ row_count │      min_time       │      max_time       │
│   int64   │      timestamp      │      timestamp      │
├───────────┼─────────────────────┼─────────────────────┤
│  67501979 │ 2019-11-01 00:00:00 │ 2019-11-30 23:59:59 │
└───────────┴─────────────────────┴─────────────────────┘



In [37]:
# 11월 고유 사용자 수와 고유 세션 수 확인
duckdb.sql(f"""
    SELECT
        COUNT(DISTINCT user_id) AS unique_users,
        COUNT(DISTINCT user_session) AS unique_sessions
    FROM read_csv_auto('{nov_path}')
""").show()

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

┌──────────────┬─────────────────┐
│ unique_users │ unique_sessions │
│    int64     │      int64      │
├──────────────┼─────────────────┤
│      3696117 │        13776050 │
└──────────────┴─────────────────┘



In [40]:
# 10월 CSV를 분석 속도가 빠른 Parquet 형식으로 변환
duckdb.sql(f"""
    COPY (
        SELECT *
        FROM read_csv_auto('{csv_path}')
    )
    TO '../data/processed/2019-Oct.parquet'
    (FORMAT PARQUET);
""")

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

In [42]:
# 11월 CSV 역시 분석 속도가 빠른 Parquet 형식으로 변환
duckdb.sql(f"""
    COPY (
        SELECT *
        FROM read_csv_auto('{nov_path}')
    )
    TO '../data/processed/2019-Nov.parquet'
    (FORMAT PARQUET);
""")

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

In [44]:
# 10월·11월 Parquet 파일을 한 번에 읽기 위한 경로 설정
parquet_path = r"..\data\processed\2019-*.parquet"

In [46]:
# 10월+11월 통합 데이터의 전체 이벤트 수와 기간 확인
duckdb.sql(f"""
    SELECT
        COUNT(*) AS total_events,
        MIN(event_time) AS min_time,
        MAX(event_time) AS max_time
    FROM read_parquet('{parquet_path}')
""").show()

┌──────────────┬─────────────────────┬─────────────────────┐
│ total_events │      min_time       │      max_time       │
│    int64     │      timestamp      │      timestamp      │
├──────────────┼─────────────────────┼─────────────────────┤
│    109950743 │ 2019-10-01 00:00:00 │ 2019-11-30 23:59:59 │
└──────────────┴─────────────────────┴─────────────────────┘



In [48]:
# 10월과 11월 전체에서 고유 사용자 수와 고유 세션 수 확인
duckdb.sql(f"""
    SELECT
        COUNT(DISTINCT user_id) AS unique_users,
        COUNT(DISTINCT user_session) AS unique_sessions
    FROM read_parquet('{parquet_path}')
""").show()

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

┌──────────────┬─────────────────┐
│ unique_users │ unique_sessions │
│    int64     │      int64      │
├──────────────┼─────────────────┤
│      5316649 │        23016650 │
└──────────────┴─────────────────┘



In [50]:
# 10월 사용자 중 11월에도 다시 활동한 사용자 수 확인
duckdb.sql(f"""
    WITH oct_users AS (
        SELECT DISTINCT user_id
        FROM read_parquet('{parquet_path}')
        WHERE event_time >= '2019-10-01'
          AND event_time < '2019-11-01'
    ),
    nov_users AS (
        SELECT DISTINCT user_id
        FROM read_parquet('{parquet_path}')
        WHERE event_time >= '2019-11-01'
          AND event_time < '2019-12-01'
    )
    SELECT
        COUNT(*) AS returning_users
    FROM oct_users o
    INNER JOIN nov_users n
        ON o.user_id = n.user_id
""").show()

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

┌─────────────────┐
│ returning_users │
│      int64      │
├─────────────────┤
│         1401758 │
└─────────────────┘



In [52]:
# 10월 활동 사용자 중 11월에도 활동한 사용자의 비율 계산
duckdb.sql(f"""
    WITH oct_users AS (
        SELECT DISTINCT user_id
        FROM read_parquet('{parquet_path}')
        WHERE event_time >= '2019-10-01'
          AND event_time < '2019-11-01'
    ),
    nov_users AS (
        SELECT DISTINCT user_id
        FROM read_parquet('{parquet_path}')
        WHERE event_time >= '2019-11-01'
          AND event_time < '2019-12-01'
    )
    SELECT
        COUNT(*) AS returning_users,
        (SELECT COUNT(*) FROM oct_users) AS oct_users,
        ROUND(
            COUNT(*) * 100.0 /
            (SELECT COUNT(*) FROM oct_users),
            2
        ) AS revisit_rate_pct
    FROM oct_users o
    INNER JOIN nov_users n
        ON o.user_id = n.user_id
""").show()

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

┌─────────────────┬───────────┬──────────────────┐
│ returning_users │ oct_users │ revisit_rate_pct │
│      int64      │   int64   │      double      │
├─────────────────┼───────────┼──────────────────┤
│         1401758 │   3022290 │            46.38 │
└─────────────────┴───────────┴──────────────────┘



In [54]:
# 확인 결과
# - 10~11월 전체 약 1억 995만 건의 이벤트, 약 532만 명의 사용자 존재
# - 10월 사용자 302만 명 중 약 140만 명이 11월에도 활동
# - 10월→11월 재방문율은 46.38%
# - 월별 사용자 수와 통합 고유 사용자 수가 논리적으로 일치 → user_id 연결 정상

In [56]:
# 10월 구매 고객 중 11월에도 다시 구매한 고객 수와 재구매율 확인
duckdb.sql(f"""
    WITH oct_buyers AS (
        SELECT DISTINCT user_id
        FROM read_parquet('{parquet_path}')
        WHERE event_time >= '2019-10-01'
          AND event_time < '2019-11-01'
          AND event_type = 'purchase'
    ),
    nov_buyers AS (
        SELECT DISTINCT user_id
        FROM read_parquet('{parquet_path}')
        WHERE event_time >= '2019-11-01'
          AND event_time < '2019-12-01'
          AND event_type = 'purchase'
    )
    SELECT
        COUNT(*) AS repeat_buyers,
        (SELECT COUNT(*) FROM oct_buyers) AS oct_buyers,
        ROUND(
            COUNT(*) * 100.0 /
            (SELECT COUNT(*) FROM oct_buyers),
            2
        ) AS repurchase_rate_pct
    FROM oct_buyers o
    INNER JOIN nov_buyers n
        ON o.user_id = n.user_id
""").show()

┌───────────────┬────────────┬─────────────────────┐
│ repeat_buyers │ oct_buyers │ repurchase_rate_pct │
│     int64     │   int64    │       double        │
├───────────────┼────────────┼─────────────────────┤
│         91286 │     347118 │                26.3 │
└───────────────┴────────────┴─────────────────────┘



In [58]:
# 확인 결과
# - 10월 구매 고객 347,118명 중 91,286명이 11월에도 구매
# - 10월→11월 재구매율은 26.30%
# - 재방문율 46.38%보다 낮아, 다시 방문해도 모두 재구매로 이어지는 것은 아님